In [1]:
import math
import numpy as np


# Production Profit Optimizer

### Mathematical Optimization of Production and Pricing Under Uncertain Demand

## 1. Project Overview

Small businesses often need to decide how many units of a product to order and what selling price to set. These decisions are important because production costs must be paid even when some products remain unsold.

This project develops a mathematical model to help businesses explore production and pricing decisions under a limited budget and uncertain customer demand.

The model is designed for businesses that purchase finished products from external manufacturers and then sell them to customers.

A protein candy business is used as the main practical example. However, the mathematical approach can also be applied to other products by changing the input parameters.



## 2. Problem Statement and Objectives

The main goal of this project is to determine the production quantity and selling price that maximize expected profit while respecting business constraints.

The model considers the following factors:

- Available production budget.
- Fixed and variable production costs.
- Minimum order quantity (MOQ).
- A range of possible selling prices.
- Customer demand estimates at two different prices.
- Uncertainty in actual customer demand.

### Research Questions

The project aims to answer four main questions:

1. What production quantities are feasible given the budget and MOQ?
2. Which combination of production quantity and selling price maximizes profit under expected demand?
3. How does uncertainty in customer demand affect expected profit and the probability of financial loss?
4. How sensitive are the results to changes in costs and demand assumptions?

### Project Approach

First, a deterministic mathematical model will be developed to analyze production costs, demand, revenue, and profit.

Next, mathematical optimization will be used to identify the most profitable feasible decisions.

Finally, Monte Carlo simulations will be performed to estimate expected profit and financial risk under uncertain demand.

The results will be validated through mathematical checks, numerical experiments, and visualizations.



## 3. Assumptions and Scope

To keep the mathematical model understandable and manageable, the following assumptions are made:

1. **Single product:** The model analyzes one product at a time. All quantities refer to finished units ready for sale.

2. **External manufacturing:** The business purchases finished products from an external manufacturer rather than producing them itself.

3. **Constant unit cost:** The production cost per unit remains constant within the analyzed order. Quantity discounts are not considered.

4. **Fixed costs:** Fixed costs do not change with the ordered quantity.

5. **Budget and MOQ:** Production decisions must satisfy the available budget and the manufacturer's minimum order quantity.

6. **Price-dependent demand:** Expected customer demand is approximated by a linear function derived from two price-demand estimates provided by the user.

7. **Single sales period:** All sales are evaluated within one period. Unsold products generate no revenue or residual value during that period.

8. **Uncertain demand:** Actual demand may differ from expected demand. The variability is represented by an explicitly stated probability model.

9. **Synthetic data:** The main experiments use illustrative input values rather than verified historical sales data.

10. **Simplified financial model:** Taxes, financing costs, storage costs, and other expenses not explicitly included in the input parameters are outside the model.

### Scope and Limitations

The project focuses on mathematical modeling, constrained profit optimization, numerical experiments, and Monte Carlo simulation.

It does not attempt to forecast actual market demand, predict future business performance, or provide guaranteed financial recommendations.

The results are conditional on the input data and modeling assumptions. Different assumptions may lead to different optimal decisions.



## 4. Mathematical Model

### 4.1. Input Parameters

The model begins with four basic production parameters: the available budget, fixed costs, unit production cost, and minimum order quantity (MOQ).

The values below represent a synthetic example of a small business purchasing finished products from an external manufacturer.

All monetary values are expressed in euros (EUR).


In [2]:

# Basic production parameters
budget = 5000
fixed_cost = 500
unit_cost = 1.50
moq = 1000

print(f"Available budget: {budget} EUR")
print(f"Fixed costs: {fixed_cost} EUR")
print(f"Unit cost: {unit_cost} EUR")
print(f"Minimum order quantity: {moq} units")


Available budget: 5000 EUR
Fixed costs: 500 EUR
Unit cost: 1.5 EUR
Minimum order quantity: 1000 units



### 4.2. Production Cost Model

Total production cost consists of fixed costs and variable costs.

The cost function is:

$$
C(q) = F + cq
$$

where:

- $C(q)$ is the total cost in EUR.
- $F$ is the fixed cost in EUR.
- $c$ is the cost per unit in EUR.
- $q$ is the ordered quantity.

The model assumes that fixed costs and unit costs remain constant within the analyzed order.


In [3]:
# Example: verify the production cost formula
example_quantity = 3000

example_total_cost = fixed_cost + unit_cost * example_quantity

print(f"Cost for {example_quantity} units: {example_total_cost:.2f} EUR")

Cost for 3000 units: 5000.00 EUR



### 4.3. Budget and Minimum Order Quantity Constraints

The business cannot order an unlimited number of units. The order must satisfy two constraints.

**Budget constraint:**

$$
F + cq \leq B
$$

**Minimum order quantity constraint:**

$$
q \geq MOQ
$$

where:

- $B$ is the available budget.
- $F$ is the fixed cost.
- $c$ is the unit production cost.
- $q$ is the ordered quantity.
- $MOQ$ is the manufacturer's minimum order quantity.

The maximum affordable quantity is:

$$
q_{\max} = \left\lfloor \frac{B-F}{c} \right\rfloor
$$

The floor operation ensures that the quantity is a whole number.

A feasible order exists only if:

$$
q_{\max} \geq MOQ
$$

For the example parameters, the feasible quantity range is from 1,000 to 3,000 units.

The maximum affordable quantity is not necessarily the most profitable quantity. The optimal order will be determined later using the profit model.


In [4]:
max_quantity = int((budget - fixed_cost) // unit_cost)

if max_quantity >= moq:
    print(max_quantity)
else:
    print("Insufficient budget to meet MOQ.")

3000


The cost formula is implemented as a reusable Python function, allowing the model to calculate production costs for different order quantities.

In [5]:
def calculate_total_cost(quantity):
    total_cost = fixed_cost + unit_cost * quantity

    return total_cost

assert calculate_total_cost(1000) == 2000
assert calculate_total_cost(3000) == 5000

print("All production cost tests passed successfully!")

All production cost tests passed successfully!



### 4.4. Customer Demand Model

Customer demand is influenced by the selling price. The model assumes a linear relationship between price and expected demand within the analyzed price range.

The demand function is:

$$
D(p) = a + bp
$$

where:
- $D(p)$ is the expected customer demand in units.
- $p$ is the selling price per unit in EUR.
- $a$ is the intercept of the demand function.
- $b$ is the change in expected demand for a 1 EUR increase in price.

The coefficients are calculated from two user-provided price-demand estimates:

$$
b = \frac{D_2-D_1}{p_2-p_1}
$$

$$
a = D_1-bp_1
$$

The two prices must be different to avoid division by zero.

For the synthetic example:
- At 3 EUR, expected demand is 2,000 units.
- At 5 EUR, expected demand is 1,000 units.

Therefore:

$$
b = \frac{1000-2000}{5-3} = -500
$$

$$
a = 2000-(-500 \cdot 3) = 3500
$$

The resulting demand function is:

$$
D(p) = 3500-500p
$$

This relationship is an assumption based on the provided estimates, not a guarantee of actual sales. The model is applied only within the price range defined by the two estimates.


In [6]:
price_1 = 3.0
demand_1 = 2000

price_2 = 5.0
demand_2 = 1000

if price_1 <= 0 or price_2 <= 0:
    raise ValueError("The selling prices must be positive.")

if demand_1 < 0 or demand_2 < 0:
    raise ValueError("Expected demand cannot be negative.")

if price_1 == price_2:
    raise ValueError("The two prices must be different.")

b = (demand_2 - demand_1) / (price_2 - price_1)

if b > 0:
    raise ValueError("Expected demand must not increase as price increases.")

a = demand_1 - (b * price_1)

print(f"Demand intercept (a): {a:.2f}")
print(f"Demand slope (b): {b:.2f}")


Demand intercept (a): 3500.00
Demand slope (b): -500.00


In [7]:
def calculate_demand(price):
    demand = a + (b * price)

    return demand

# Validate the customer demand model
assert calculate_demand(price_1) == demand_1
assert calculate_demand(price_2) == demand_2

print("Customer demand model tests passed successfully!")

Customer demand model tests passed successfully!



#### Input Validation and Model Assumptions

The input parameters are validated before calculating the demand function.

The following conditions must be satisfied:

- Both selling prices must be positive.
- Expected demand values must be non-negative.
- The two selling prices must be different.
- Expected demand must not increase as the selling price increases.

For the linear demand function:

$$
D(p) = a + bp
$$

the final condition requires:

$$
b \leq 0
$$

Invalid inputs raise a `ValueError` with an explanatory message.

These restrictions ensure that the input data is consistent with the assumptions of the selected demand model.

The model is valid only within the price interval defined by the two input estimates. Predictions outside this interval are not considered in the optimization.



### 4.5. Revenue and Profit Model

The number of units sold cannot exceed either the available stock or customer demand.

Expected sales under the deterministic demand model are:

$$
S(q,p) = \min(q,D(p))
$$

where:
- $q$ is the ordered quantity.
- $p$ is the selling price per unit.
- $D(p)$ is expected customer demand.
- $S(q,p)$ is the expected quantity sold under the deterministic demand assumption.

Revenue is calculated as:

$$
R(q,p) = p \cdot S(q,p)
$$

Total profit is revenue minus production costs:

$$
P(q,p) = R(q,p) - C(q)
$$

Combining the formulas:

$$
P(q,p) = p \cdot \min(q,D(p)) - (F+cq)
$$

The model assumes that unsold units do not generate revenue during the analyzed period, while their production costs are still incurred.

#### Example

For an order of 1,800 units at a selling price of 4 EUR:

- Expected demand: 1,500 units.
- Expected sales: 1,500 units.
- Revenue: 6,000 EUR.
- Total costs: 3,200 EUR.
- Expected profit: 2,800 EUR.

This profit is calculated under deterministic expected demand. Demand uncertainty will be introduced later.


In [8]:
def calculate_sales(quantity, price):
    demand = calculate_demand(price)
    sales = min(quantity, demand)

    return sales

assert calculate_sales(1000, 4) == 1000
assert calculate_sales(1800, 4) == 1500

In [9]:
def calculate_revenue(quantity, price):
    sales = calculate_sales(quantity, price)
    revenue = sales * price 

    return revenue

assert calculate_revenue(1800, 4) == 6000

In [10]:
def calculate_profit(quantity, price):
    revenue = calculate_revenue(quantity, price)
    total_cost = calculate_total_cost(quantity)
    profit = revenue - total_cost

    return profit

assert calculate_profit(1800, 4) == 2800
assert calculate_profit(2000, 4) == 2500

print("Profit model tests passed successfully!")

Profit model tests passed successfully!



### 4.6. Deterministic Profit Optimization

The objective is to find the selling price and order quantity that maximize profit under the assumed demand model.

The objective function is:

$$
\max_{q,p} P(q,p)
= p \cdot \min(q,D(p)) - (F+cq)
$$

subject to:

$$
MOQ \leq q \leq q_{\max}, \quad q \in \mathbb{Z}
$$

$$
p_{\min} \leq p \leq p_{\max}
$$

where the price limits are determined by the two user-provided price-demand estimates.

#### Optimization Method: Grid Search

The algorithm evaluates selling prices from the minimum to the maximum input price in increments of 0.01 EUR.

For each candidate price:

1. Calculate the expected customer demand.
2. Determine the most profitable feasible integer order quantity.
3. Calculate the corresponding profit.
4. Compare the result with the best solution found so far.

For a fixed price above the unit cost, the optimal quantity can be found by comparing the feasible integer quantities immediately below and above expected demand, after applying the MOQ and budget limits.

If the selling price does not exceed the unit cost, the model chooses MOQ, assuming an order is mandatory.

The model assumes deterministic demand, constant unit costs, and no revenue from unsold units during the analyzed period.

The best result is selected from the evaluated price grid. Therefore, the result is optimal for the tested prices, but it is not yet an analytical proof of a continuous global maximum.


In [11]:
def calculate_optimal_quantity(price):
    demand = calculate_demand(price)

    if moq > max_quantity:
        raise ValueError("The budget is insufficient to meet the minimum order quantity (MOQ).")

    if price <= unit_cost:
        return moq

    lower_quantity = math.floor(demand)
    upper_quantity = math.ceil(demand)

    lower_quantity = max(moq, min(lower_quantity, max_quantity))
    upper_quantity = max(moq, min(upper_quantity, max_quantity))

    lower_profit = calculate_profit(lower_quantity, price)
    upper_profit = calculate_profit(upper_quantity, price)

    if lower_profit < upper_profit:
        best_quantity = upper_quantity
    else:
        best_quantity = lower_quantity

    return best_quantity

assert calculate_optimal_quantity(4) == 1500
assert calculate_optimal_quantity(5) == 1000
assert calculate_optimal_quantity(3.9988) == 1501
assert calculate_optimal_quantity(1.20) == 1000

print("Optimal quantity tests passed successfully!")

Optimal quantity tests passed successfully!


In [12]:
best_profit = float("-inf")
best_price = None
best_quantity = None

min_price = min(price_1, price_2)
max_price = max(price_1, price_2)

for price_cents in range(round(min_price * 100), round(max_price * 100) + 1):
    price = price_cents / 100

    optimal_quantity = calculate_optimal_quantity(price)
    profit = calculate_profit(optimal_quantity, price)

    if profit > best_profit:
        best_profit = profit
        best_price = price
        best_quantity = optimal_quantity

print(f"Optimal selling price: {best_price:.2f} EUR")
print(f"Optimal order quantity: {best_quantity} units")
print(f"Maximum profit: {best_profit:.2f} EUR")

assert best_quantity >= moq
assert best_quantity <= max_quantity
assert calculate_total_cost(best_quantity) <= budget
assert min_price <= best_price <= max_price

Optimal selling price: 4.25 EUR
Optimal order quantity: 1375 units
Maximum profit: 3281.25 EUR



### 4.7. Analytical Verification Using Derivatives

To verify the result of the deterministic grid search, we use calculus to find the maximum of the profit function analytically.

For a selling price above the unit cost, and assuming that the optimal order quantity equals expected demand and satisfies the business constraints:

$$
q = D(p)
$$

The profit function becomes:

$$
P(p) = (p-c)D(p)-F
$$

Since expected demand is linear:

$$
D(p) = a+bp
$$

we obtain:

$$
P(p) = (p-c)(a+bp)-F
$$

Expanding the expression:

$$
P(p) = bp^2+(a-bc)p-(ac+F)
$$

For our example:

$$
P(p) = -500p^2+4250p-5750
$$

**First derivative:**

$$
P'(p) = -1000p+4250
$$

Setting the first derivative to zero:

$$
-1000p+4250 = 0
$$

$$
p^* = 4.25
$$

**Second derivative:**

$$
P''(p) = -1000 < 0
$$

Therefore, the critical point is the global maximum of this quadratic profit function.

The corresponding expected demand is:

$$
q^* = D(4.25) = 1375
$$

This quantity satisfies the minimum order quantity and budget constraints. The optimal price is also within the allowed price range.

The resulting maximum expected profit is:

$$
P^* = 3281.25 \text{ EUR}
$$

The analytical result matches the deterministic grid search result for the example inputs.

**Limitation:** This analytical approach assumes that ordering the expected demand is optimal and feasible. If the price, budget, or MOQ constraints are binding, the constrained optimum may differ. The grid search remains the primary optimization method.


In [13]:
quadratic_a = b
quadratic_b = a - (b * unit_cost)
analytical_price = -(quadratic_b / (2*quadratic_a))

analytical_quantity = calculate_demand(analytical_price)

second_derivative = 2 * quadratic_a
assert second_derivative < 0

print(analytical_quantity)
print(analytical_price)

assert abs(analytical_price - best_price) < 1e-9
assert abs(analytical_quantity - best_quantity) < 1e-9

1375.0
4.25



### 4.8. Stochastic Demand Model and Monte Carlo Simulation

#### Stochastic Demand

Unlike the deterministic model, which assumes a fixed expected demand for each selling price, the stochastic model accounts for uncertainty in customer demand.

Expected demand is still calculated using the linear function:

$$
D(p) = a + bp
$$

We assume that actual customer demand approximately follows a normal distribution:

$$
X_p \sim \mathcal{N}(D(p), \sigma(p)^2)
$$

where:

- $X_p$ is random customer demand at selling price $p$.
- $D(p)$ is the expected demand.
- $\sigma(p)$ is the standard deviation of demand.

The standard deviation is modeled as:

$$
\sigma(p) = r \cdot D(p)
$$

where $r$ is the uncertainty rate.

For the initial scenario, we assume $r = 0.20$, meaning that the standard deviation equals 20% of expected demand.

This percentage is a modeling assumption, not a value estimated from historical sales data. Users can explore different uncertainty rates in sensitivity analysis.

Since the number of products demanded cannot be negative or fractional, simulated demand is rounded to the nearest integer and limited to a minimum of zero.

#### Stochastic Profit

For an order quantity $q$, selling price $p$, and randomly generated demand $X_p$, actual sales are:

$$
S(q,p,X_p) = \min(q,X_p)
$$

The simulated profit is:

$$
P(q,p,X_p) = p \cdot \min(q,X_p) - (F+cq)
$$

Unsold products remain a cost to the business. The model does not assume additional revenue from unsold inventory.

#### Monte Carlo Simulation

Monte Carlo simulation evaluates a business decision under uncertainty by generating many independent random demand scenarios.

For each simulation:

1. Generate random customer demand.
2. Determine the number of products sold.
3. Calculate revenue and total cost.
4. Calculate and store the simulated profit.

We use 10,000 simulations for the initial analysis.

#### Risk Analysis

The simulation results are evaluated using three metrics.

**Average Simulated Profit**

$$
\overline{P} = \frac{1}{N}\sum_{i=1}^{N}P_i
$$

This estimates the expected profit under the assumed stochastic demand model.

**Probability of Loss**

$$
\widehat{P}(\text{Loss}) =
\frac{\#\{P_i < 0\}}{N}
$$

This estimates the fraction of simulated scenarios resulting in a negative profit.

**Standard Deviation of Profit**

$$
\sigma_P =
\sqrt{\frac{1}{N}\sum_{i=1}^{N}(P_i-\overline{P})^2}
$$

This measures how widely simulated profits vary around their average.

#### Limitations

The results depend on the assumed demand model, uncertainty rate, and other input parameters. They are estimates rather than guarantees of future sales or profitability.

Rounding and clipping simulated demand to non-negative integers also modify the theoretical normal distribution.

This stage evaluates a selected price and order quantity. Optimization under uncertainty will be implemented separately.


In [14]:
uncertainty_rate = 0.20

def calculate_demand_std(price):
    demand = calculate_demand(price)
    std = uncertainty_rate * demand

    return std

assert calculate_demand_std(4.25) == 275.0
assert calculate_demand_std(5) == 200.0


In [15]:
def generate_random_demand(price):
    demand = calculate_demand(price)
    demand_std = calculate_demand_std(price)
    random_demand = round(np.random.normal(loc=demand, scale=demand_std))

    return max(0, random_demand)


random_demand = generate_random_demand(4.25)

assert isinstance(random_demand, int)
assert random_demand >= 0

print(f"Generated demand: {random_demand} units")

Generated demand: 1477 units


In [16]:
def calculate_stochastic_profit(quantity, price):
    random_demand = generate_random_demand(price)
    total_cost = calculate_total_cost(quantity)

    stochastic_profit = price * (min(quantity,random_demand)) - total_cost

    return stochastic_profit

simulated_profit = calculate_stochastic_profit(1375, 4.25)
assert simulated_profit <= calculate_profit(1375, 4.25)

In [17]:
def monte_carlo_simulation(quantity, price, num_simulations):
    profits = []
    for n in range(num_simulations):
        stochastic_profit = calculate_stochastic_profit(quantity, price)
        profits.append(stochastic_profit)

    return profits

profits = monte_carlo_simulation(1375, 4.25, 10000)

In [18]:
def analyze_simulation(profits):

    average_profit = sum(profits) / len(profits)

    loss_count = 0
    for p in profits:
        if p < 0:
            loss_count += 1

    loss_probability = loss_count / len(profits)

    profit_std = np.std(profits)

    return average_profit, loss_probability, profit_std


In [19]:
average_profit, loss_probability, profit_std = analyze_simulation(profits)

print(f"Number of simulations: {len(profits)}")
print(f"Average simulated profit: {average_profit:.2f} EUR")
print(f"Probability of loss: {loss_probability * 100:.2f}%")
print(f"Standard deviation of profit: {profit_std:.2f} EUR")

Number of simulations: 10000
Average simulated profit: 2813.93 EUR
Probability of loss: 0.27%
Standard deviation of profit: 684.64 EUR
